# 02 — LAPD Dataset Cleaning

**PACE Phase**: Analyze
**Objective**: apply full cleaning to the combined dataset and produce
`crimes_clean.parquet`, the reference base for all subsequent notebooks.

**Input**: `data/processed/crimes_merged.parquet` (produced by `01_data_loading.ipynb`)
**Output**: `data/processed/crimes_clean.parquet`

## 1. Setup and loading

In [1]:
import pandas as pd  # Import the Pandas framework for DataFrame manipulation
import numpy as np   # Import the Numpy framework to handle numeric operations

pd.set_option('display.max_rows', None)         # Pandas setting to show all rows with inspection commands
pd.set_option('display.max_columns', None)      # Pandas setting to show all columns with inspection commands
pd.set_option('display.max_info_columns', 200)  # Pandas setting to show info for 200 columns with the '.info()' command

df = pd.read_parquet('../../data/processed/crimes_merged.parquet') # Create the variable 'df' containing the data
                                                                   # from the parquet file 'crimes_merged.parquet'

print(f"Dataset loaded: {df.shape}")                             # Display the number of rows and columns of the DataFrame

print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")  # Display the amount of memory used to run
                                                                        # the previous operations

Dataset loaded: (3138031, 28)
Memory: 2659.9 MB


## 2. Dropping irrelevant columns

Based on the initial inspection (Plan phase), the columns that won't add value to the analyses are dropped:

- **`Crm Cd 2`, `Crm Cd 3`, `Crm Cd 4`** (>93% null): represent secondary
  crimes linked to the main report, but are practically always empty.
  For our analytical questions, `Crm Cd` (primary crime) is sufficient.
- **`Cross Street`** (84% null): information redundant with `LOCATION`
  and the `LAT`/`LON` coordinates.

In [2]:
colonne_da_eliminare = ['Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'Cross Street'] # List of the columns to drop, decided
                                                                           # in the previous markdown cell

df = df.drop(columns=colonne_da_eliminare) # '.drop' removes the listed columns from the DataFrame

print(f"Dropped columns: {colonne_da_eliminare}")   # Display the list of dropped columns
print(f"New shape: {df.shape}")                     # Display the DataFrame's shape after the drop
print(f"Remaining columns: {df.shape[1]}")          # Display the number of remaining columns

Dropped columns: ['Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'Cross Street']
New shape: (3138031, 24)
Remaining columns: 24


## 3. Data type conversion

### 3.1 Dates (`Date Rptd` and `DATE OCC`)

Both columns are currently strings in the format `MM/DD/YYYY HH:MM:SS AM/PM`
(US format). They are converted to `datetime64` to allow temporal operations, 
filters by year/month, interval calculations, etc.

- `DATE OCC` = date the crime occurred (the most important for the analyses)
- `Date Rptd` = date the crime was reported/recorded (may be
  later than the occurrence date, and the delta is an interesting piece of information)

In [3]:
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')   # Convert the 'DATE OCC' column to datetime format
df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], errors='coerce') # Convert the 'Date Rptd' column to datetime format

print("Types after conversion:")  # Display the given text

print(df[['DATE OCC', 'Date Rptd']].dtypes)  # Display the dtype of the 'DATE OCC' and 'Date Rptd' columns

print(f"\nNull DATE OCC: {df['DATE OCC'].isnull().sum()}")  # Show the number of null values in the 'DATE OCC' column

print(f"Null Date Rptd: {df['Date Rptd'].isnull().sum()}")  # Show the number of null values in the 'Date Rptd' column

print(f"\nRange DATE OCC: {df['DATE OCC'].min()} → {df['DATE OCC'].max()}")  # Display the min and max values of the 'DATE OCC' column
print(f"Range Date Rptd: {df['Date Rptd'].min()} → {df['Date Rptd'].max()}") # Display the min and max values of the 'Date Rptd' column

/var/folders/wl/_8d4j_cs57qgchs9ckhqzkyr0000gn/T/ipykernel_4714/978203801.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')   # Convert the 'DATE OCC' column to datetime format
/var/folders/wl/_8d4j_cs57qgchs9ckhqzkyr0000gn/T/ipykernel_4714/978203801.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], errors='coerce') # Convert the 'Date Rptd' column to datetime format


Types after conversion:
DATE OCC     datetime64[ns]
Date Rptd    datetime64[ns]
dtype: object

Null DATE OCC: 0
Null Date Rptd: 0

Range DATE OCC: 2010-01-01 00:00:00 → 2024-12-30 00:00:00
Range Date Rptd: 2010-01-01 00:00:00 → 2025-06-05 00:00:00


### 3.2 Time (`TIME OCC`)

`TIME OCC` is currently an integer in HHMM format (e.g. `2130` = 21:30,
`45` = 00:45, `0` = 00:00). We transform it into something more manageable:
an `hour_occ` column with only the integer hour (0-23), which is the
granularity sufficient for analyses such as "most critical time slots" or
"hourly distribution of crimes".

The original `TIME OCC` column is replaced.

In [4]:
df['hour_occ'] = df['TIME OCC'] // 100 # Apply the 'integer division' operator to the 'TIME OCC' column to extract
                                       # the hour from the HHMM integer, creating the 'hour_occ' column

print(f"Range hour_occ: {df['hour_occ'].min()} → {df['hour_occ'].max()}") # Check operation displaying the min and max numbers
                                                                          # of the 'hour_occ' column

print(f"\nDistribution by hour (top 5):")                                # Display the given string

print(df['hour_occ'].value_counts().sort_index().head(5))                 # Display the first 5 rows of the 'hour_occ' column
                                                                          # sorted in ascending order by index

print(f"\nOut-of-range values (>23 or <0): {((df['hour_occ'] < 0) | (df['hour_occ'] > 23)).sum()}")
# Display the given f-string containing text and the sum of 'hour_occ' values below 0 and above 23
# to check that all values fall within the min/max range created earlier


Range hour_occ: 0 → 23

Distribution by hour (top 5):
hour_occ
0    130102
1     90502
2     76769
3     60937
4     48319
Name: count, dtype: int64

Out-of-range values (>23 or <0): 0


In [5]:
df = df.drop(columns=['TIME OCC'])                              # '.drop' removes the 'TIME OCC' column, now replaced by 'hour_occ'
print(f"Column 'TIME OCC' dropped. Current shape: {df.shape}") # Display the DataFrame's shape after the drop

Column 'TIME OCC' dropped. Current shape: (3138031, 24)


### 3.3 Handling sentinel coordinates

In notebook 01 we identified 3,148 records with `(0, 0)` coordinates,
which the LAPD uses as a sentinel value to indicate "unknown location".

**NOTE:** 
As documented in the LAPD dataset description (data.lacity.org): 'Some location fields with missing data are noted as (0°, 0°). Address fields are only provided to the nearest hundred block in order to maintain privacy

They are replaced with `NaN` instead of dropping the records: this way the records remain available
for non-geographic analyses (crime type, victims,
time of day) but are automatically excluded from analyses that require
the coordinates.

In [6]:
maschera_sentinella = (df['LAT'] == 0) & (df['LON'] == 0) # Create a boolean mask that is True where both
                                                          # LAT and LON are the sentinel value 0
n_sentinelle = maschera_sentinella.sum()                  # Count the records matching the sentinel mask

df.loc[maschera_sentinella, ['LAT', 'LON']] = np.nan      # Replace the sentinel coordinates with NaN so the
                                                          # records stay available for non-geographic analyses

print(f"Records with sentinel coordinates replaced: {n_sentinelle}")     # Display the count just computed
print(f"Null LAT after replacement: {df['LAT'].isnull().sum()}")         # Check the new null count for LAT
print(f"Null LON after replacement: {df['LON'].isnull().sum()}")         # Check the new null count for LON
print(f"\nNew LAT range: {df['LAT'].min():.4f} → {df['LAT'].max():.4f}")  # Display the LAT range after cleaning
print(f"New LON range: {df['LON'].min():.4f} → {df['LON'].max():.4f}")   # Display the LON range after cleaning

Records with sentinel coordinates replaced: 3148
Null LAT after replacement: 3148
Null LON after replacement: 3148

New LAT range: 33.3427 → 34.7907
New LON range: -118.8279 → -117.6596


## 4. Handling duplicates on DR_NO

`DR_NO` (Division of Records Number) is the unique identifier of the police
report. In theory there shouldn't be any duplicates, but in the initial
inspection we found 57,809.

Before deciding how to handle them, we inspect a few real cases to understand
**why** they exist and how the duplicate rows differ.

In [7]:
dr_no_duplicati = df[df.duplicated(subset='DR_NO', keep=False)]['DR_NO'].unique() # '.duplicated(keep=False)' flags all rows
                                                                                  # involved in a duplicate on 'DR_NO';
                                                                                  # '.unique()' keeps only the distinct DR_NO values
print(f"Unique duplicated DR_NO: {len(dr_no_duplicati)}")                        # Display the number of distinct duplicated DR_NO
print(f"Total rows involved: {df.duplicated(subset='DR_NO', keep=False).sum()}") # Display the total rows flagged as duplicates

print("\n=== First 3 examples of duplicates ===\n") # Display the given string

for dr_no in dr_no_duplicati[:3]:               # Iterate over the first 3 duplicated DR_NO values
    print(f"--- DR_NO: {dr_no} ---")           # Display a header with the current DR_NO
    print(df[df['DR_NO'] == dr_no])            # Display all the rows sharing that DR_NO, to compare them
    print()

Unique duplicated DR_NO: 57809
Total rows involved: 115618

=== First 3 examples of duplicates ===

--- DR_NO: 161804259 ---
             DR_NO  Date Rptd   DATE OCC  AREA  AREA NAME  Rpt Dist No  \
937560   161804259 2016-01-06 2016-01-06    18  Southeast         1805   
1234823  161804259 2016-01-06 2016-01-06    18  Southeast         1805   

         Part 1-2  Crm Cd       Crm Cd Desc Mocodes  Vict Age Vict Sex  \
937560          1     510  VEHICLE - STOLEN    None         0     None   
1234823         1     510  VEHICLE - STOLEN    None         0     None   

        Vict Descent  Premis Cd Premis Desc  Weapon Used Cd Weapon Desc  \
937560          None      101.0      STREET             NaN        None   
1234823         None      101.0      STREET             NaN        None   

        Status  Status Desc  Crm Cd 1                                LOCATION  \
937560      IC  Invest Cont     510.0  200 E  90TH                         ST   
1234823     IC  Invest Cont     510.0  20

In [8]:
duplicati_su_dr_no = df.duplicated(subset='DR_NO', keep=False).sum() # Count rows flagged as duplicates on 'DR_NO' only
duplicati_esatti_riga = df.duplicated(keep=False).sum()              # Count rows flagged as duplicates across
                                                                     # every column (exact duplicates)

print(f"Rows with duplicated DR_NO: {duplicati_su_dr_no}")             # Display the count of DR_NO duplicates
print(f"Exact duplicate rows (all columns): {duplicati_esatti_riga}")  # Display the count of exact duplicates

if duplicati_su_dr_no == duplicati_esatti_riga: # If the two counts match, every DR_NO duplicate is also
                                                # an exact duplicate across all columns
    print("\n✅ All DR_NO duplicates are exact duplicates")
else:                                            # Otherwise some rows share DR_NO but differ in other columns
    diff = duplicati_su_dr_no - duplicati_esatti_riga
    print(f"\n⚠️ There are {diff} rows with duplicated DR_NO but with differences in the columns")

Rows with duplicated DR_NO: 115618
Exact duplicate rows (all columns): 115618

✅ All DR_NO duplicates are exact duplicates


In [9]:
duplicati_df = df[df.duplicated(subset='DR_NO', keep=False)] # Isolate all rows involved in a DR_NO duplicate

print("Top 10 crime types among duplicates:")                                          # Display the given string
print(duplicati_df['Crm Cd Desc'].value_counts().head(10))                             # Display the 10 most frequent
                                                                                        # crime types among the duplicates
print(f"\nTotal distinct crime types among duplicates: {duplicati_df['Crm Cd Desc'].nunique()}") # Display the number
                                                                                                  # of distinct crime types involved

Top 10 crime types among duplicates:
Crm Cd Desc
THEFT OF IDENTITY                                          10608
VEHICLE - STOLEN                                            8792
BATTERY - SIMPLE ASSAULT                                    8086
BURGLARY FROM VEHICLE                                       7730
BURGLARY                                                    7698
THEFT PLAIN - PETTY ($950 & UNDER)                          6864
INTIMATE PARTNER - SIMPLE ASSAULT                           6464
VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)     5974
THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER)             5348
ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT              4726
Name: count, dtype: int64

Total distinct crime types among duplicates: 122


### Removing duplicates

All 57,809 duplicated DR_NO turn out to be exact duplicates (rows
identical across all columns), spread across 122 different crime types.

We remove the duplicates, keeping the first occurrence of each row.

In [10]:
prima = len(df)          # Store the row count before removing duplicates
df = df.drop_duplicates() # '.drop_duplicates()' removes the exact duplicate rows, keeping the first occurrence
dopo = len(df)            # Store the row count after removing duplicates

print(f"Rows before: {prima}")                                              # Display the row count before
print(f"Rows after: {dopo}")                                                # Display the row count after
print(f"Rows removed: {prima - dopo}")                                      # Display the number of rows removed
print(f"\nRemaining duplicates on DR_NO: {df['DR_NO'].duplicated().sum()}") # Check that no DR_NO duplicates remain

Rows before: 3138031
Rows after: 3080222
Rows removed: 57809

Remaining duplicates on DR_NO: 0


## 5. Handling null values

We update the null-value overview after dropping columns and duplicates.

In [11]:
nulli_pct = (df.isnull().sum() / len(df) * 100).round(2)          # Compute the percentage of null values per column
nulli_pct = nulli_pct[nulli_pct > 0].sort_values(ascending=False) # Keep only columns with nulls, sorted descending
print("Columns with null values:")                                # Display the given string
print(nulli_pct)                                                  # Display the 'nulli_pct' variable

Columns with null values:
Weapon Used Cd    66.73
Weapon Desc       66.73
Mocodes           12.18
Vict Sex          10.95
Vict Descent      10.95
LAT                0.10
LON                0.10
Premis Desc        0.03
dtype: float64


### 5.1 Weapon Used Cd / Weapon Desc

The 66.73% of null values doesn't indicate missing data: most crimes
simply don't involve a weapon (theft, fraud, vandalism, etc.).
We recode NaN as "No Weapon" in the description and as `0` in the code.

In [12]:
df['Weapon Desc'] = df['Weapon Desc'].fillna('No Weapon') # Recode null 'Weapon Desc' as "No Weapon", since the
                                                          # null values mean the crime didn't involve a weapon
df['Weapon Used Cd'] = df['Weapon Used Cd'].fillna(0)     # Recode the matching code column as 0, for consistency

print("Weapon Desc — most frequent values:")                          # Display the given string
print(df['Weapon Desc'].value_counts().head(5))                       # Display the 5 most frequent values
print(f"\nNull Weapon Desc: {df['Weapon Desc'].isnull().sum()}")      # Check that no nulls remain
print(f"Null Weapon Used Cd: {df['Weapon Used Cd'].isnull().sum()}")  # Check that no nulls remain

Weapon Desc — most frequent values:
Weapon Desc
No Weapon                                         2055380
STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)     598043
UNKNOWN WEAPON/OTHER WEAPON                         92736
VERBAL THREAT                                       81418
HAND GUN                                            53593
Name: count, dtype: int64

Null Weapon Desc: 0
Null Weapon Used Cd: 0


### 5.2 Vict Sex / Vict Descent

The ~11% of null values represents cases where the victim was not identified
or the crime has no direct victim (e.g. vehicle theft, vandalism).
We recode both columns as "X" (Unknown), consistent with
the coding already used by the LAPD for uncertain cases.

In [13]:
df['Vict Sex'] = df['Vict Sex'].fillna('X')         # Recode null 'Vict Sex' as "X" (Unknown), consistent with
                                                    # the coding already used by the LAPD
df['Vict Descent'] = df['Vict Descent'].fillna('X') # Recode null 'Vict Descent' as "X" (Unknown) as well

print("Vict Sex — distribution:")                                          # Display the given string
print(df['Vict Sex'].value_counts())                                       # Display the value counts
print(f"\nVict Descent — unique values: {df['Vict Descent'].nunique()}")   # Display the number of distinct values
print(f"Null Vict Sex: {df['Vict Sex'].isnull().sum()}")                   # Check that no nulls remain
print(f"Null Vict Descent: {df['Vict Descent'].isnull().sum()}")           # Check that no nulls remain

Vict Sex — distribution:
Vict Sex
M    1360215
F    1230645
X     489158
H        185
N         17
-          2
Name: count, dtype: int64

Vict Descent — unique values: 20
Null Vict Sex: 0
Null Vict Descent: 0


### 5.3 Mocodes

Mocodes (modus operandi codes) are null in 12% of cases.
This is a legitimate missing value: not all crimes have a coded modus
operandi. We leave them as NaN and will handle them specifically
in the analysis notebook dedicated to modus operandi patterns.

In [14]:
print(f"Mocodes — null: {df['Mocodes'].isnull().sum()} ({df['Mocodes'].isnull().mean()*100:.2f}%)") # Display the
                                                                                                     # null count and percentage
print(f"Example of a non-null Mocodes value: {df['Mocodes'].dropna().iloc[0]}")                     # Display a sample value

Mocodes — null: 375261 (12.18%)
Example of a non-null Mocodes value: 0913 1814 2000


### 5.4 Premis Desc

Only 0.03% null. Negligible percentage. We remove these few
rows because it isn't worth imputing a value and the loss is irrelevant.

In [15]:
prima = len(df)                                # Store the row count before dropping
df = df.dropna(subset=['Premis Desc'])         # '.dropna' removes the few rows with a null 'Premis Desc'
print(f"Rows removed for null Premis Desc: {prima - len(df)}") # Display the number of rows removed
print(f"Current shape: {df.shape}")                             # Display the DataFrame's shape after the drop

Rows removed for null Premis Desc: 775
Current shape: (3079447, 24)


### 5.5 Cleaning anomalous values in Vict Sex

Besides M, F, X there are rare values: `H` (185), `N` (17), `-` (2).
We recode all of them as `X` (Unknown) for simplicity, since they represent
0.007% of the dataset and are not sufficient for meaningful statistical
analysis on the category.

In [16]:
valori_validi = ['M', 'F', 'X'] # List of the only values considered valid for 'Vict Sex'

df.loc[~df['Vict Sex'].isin(valori_validi), 'Vict Sex'] = 'X' # Recode every value not in 'valori_validi'
                                                              # (rare anomalies like 'H', 'N', '-') as "X"

print("Vict Sex after cleaning:")           # Display the given string
print(df['Vict Sex'].value_counts())        # Display the value counts after recoding

Vict Sex after cleaning:
Vict Sex
M    1359854
F    1230571
X     489022
Name: count, dtype: int64


## 6. Handling sentinel values in Vict Age

In [17]:
print("Vict Age statistics:")                                           # Display the given string
print(df['Vict Age'].describe())                                        # Display descriptive statistics
print(f"\nValues <= 0: {(df['Vict Age'] <= 0).sum()}")                  # Count the sentinel/anomalous low values
print(f"Values > 120: {(df['Vict Age'] > 120).sum()}")                  # Count implausibly high values
print(f"\nDistribution of anomalous values:")                           # Display the given string
print(df[df['Vict Age'] <= 0]['Vict Age'].value_counts().sort_index()) # Display the breakdown of the values <= 0

Vict Age statistics:
count    3.079447e+06
mean     3.081706e+01
std      2.113740e+01
min     -1.300000e+01
25%      1.800000e+01
50%      3.100000e+01
75%      4.600000e+01
max      1.200000e+02
Name: Vict Age, dtype: float64

Values <= 0: 632399
Values > 120: 0

Distribution of anomalous values:
Vict Age
-13         1
-12         3
-11         2
-10        12
-9         18
-8         13
-7         18
-6         25
-5         38
-4         51
-3         79
-2        152
-1        366
 0     631621
Name: count, dtype: int64


`Vict Age` is an integer that should represent the victim's age.
However, the inspection revealed anomalous values equal to or below zero that need to be addressed.

Since no explicit convention was found in the data documentation, data will be treated as follow:
- `Vict Age = 0` (631,621 records, ~20.5%): Those values are treated as a sentinel value for "unknown age".
- `Vict Age < 0` (778 records): Those values are treated as data entry errors.

### 6.1 Recoding sentinel and anomalous values
All values ≤ 0 will be replaced with `NaN`: the records remain available for
non-demographic analyses, but are automatically excluded from
age-based calculations (mean, median, distribution, etc.).

In [18]:
maschera = df['Vict Age'] <= 0        # Create a boolean mask for the sentinel (0) and anomalous (negative) values
n_sostituiti = maschera.sum()         # Count the values matching the mask

df.loc[maschera, 'Vict Age'] = np.nan # Replace them with NaN so the records stay available for non-demographic
                                      # analyses but are excluded from age-based calculations

print(f"Values replaced with NaN: {n_sostituiti}")           # Display the count just computed
print(f"\nVict Age statistics after cleaning:")               # Display the given string
print(df['Vict Age'].describe())                              # Display descriptive statistics after cleaning
print(f"\nNull Vict Age: {df['Vict Age'].isnull().sum()}")    # Display the new null count

Values replaced with NaN: 632399

Vict Age statistics after cleaning:
count    2.447048e+06
mean     3.878205e+01
std      1.591625e+01
min      2.000000e+00
25%      2.700000e+01
50%      3.600000e+01
75%      5.000000e+01
max      1.200000e+02
Name: Vict Age, dtype: float64

Null Vict Age: 632399


## 7. Final check

Summary of the dataset's state after all cleaning operations.

In [19]:
print("=" * 50)
print("FINAL CHECK — CLEANED DATASET")
print("=" * 50)

print(f"\nShape: {df.shape}")                                          # Display rows and columns
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")  # Display memory usage
print(f"Duplicates on DR_NO: {df['DR_NO'].duplicated().sum()}")        # Confirm no DR_NO duplicates remain

print(f"\n--- Data types ---")
print(df.dtypes)

print(f"\n--- Null values ---")
nulli = df.isnull().sum()   # Count nulls per column
nulli = nulli[nulli > 0]    # Keep only columns with at least one null
for col, n in nulli.items():
    print(f"  {col}: {n} ({n/len(df)*100:.2f}%)")

if len(nulli) == 0:
    print("  No null values")

print(f"\n--- Temporal range ---")
print(f"  DATE OCC: {df['DATE OCC'].min().date()} → {df['DATE OCC'].max().date()}")
print(f"  Date Rptd: {df['Date Rptd'].min().date()} → {df['Date Rptd'].max().date()}")

print(f"\n--- Geographic range ---")
print(f"  LAT: {df['LAT'].min():.4f} → {df['LAT'].max():.4f}")
print(f"  LON: {df['LON'].min():.4f} → {df['LON'].max():.4f}")

print(f"\n--- Key counts ---")
print(f"  Years covered: {df['DATE OCC'].dt.year.nunique()}")  # Distinct years present in 'DATE OCC'
print(f"  LAPD areas: {df['AREA NAME'].nunique()}")             # Distinct LAPD areas
print(f"  Crime types: {df['Crm Cd Desc'].nunique()}")          # Distinct crime type descriptions

FINAL CHECK — CLEANED DATASET

Shape: (3079447, 24)
Memory: 2180.7 MB
Duplicates on DR_NO: 0

--- Data types ---
DR_NO                      int64
Date Rptd         datetime64[ns]
DATE OCC          datetime64[ns]
AREA                       int64
AREA NAME                 object
Rpt Dist No                int64
Part 1-2                   int64
Crm Cd                     int64
Crm Cd Desc               object
Mocodes                   object
Vict Age                 float64
Vict Sex                  object
Vict Descent              object
Premis Cd                float64
Premis Desc               object
Weapon Used Cd           float64
Weapon Desc               object
Status                    object
Status Desc               object
Crm Cd 1                 float64
LOCATION                  object
LAT                      float64
LON                      float64
hour_occ                   int64
dtype: object

--- Null values ---
  Mocodes: 375192 (12.18%)
  Vict Age: 632399 (20.54%)
  Sta

### 7.1 Residual cleaning

The final check revealed 2 nulls in `Status` and 21 in `Crm Cd 1`,
which had not emerged previously. Given the negligible amount (23 rows out of
3+ million), we remove them.

In [20]:
prima = len(df)                                    # Store the row count before dropping
df = df.dropna(subset=['Status', 'Crm Cd 1'])      # '.dropna' removes the residual nulls found in the final check
print(f"Rows removed: {prima - len(df)}")          # Display the number of rows removed
print(f"Final shape: {df.shape}")                  # Display the DataFrame's final shape

Rows removed: 23
Final shape: (3079424, 24)


## 8. Saving the cleaned dataset

The cleaned DataFrame is saved as `crimes_clean.parquet` in `data/processed/`.
This file will be the starting point for all subsequent notebooks
(feature engineering, EDA, visualizations, clustering).

In [21]:
df.to_parquet('../../data/processed/crimes_clean.parquet') # '.to_parquet' saves the cleaned DataFrame,
                                                            # the reference base for all subsequent notebooks
print("✅ Cleaned dataset saved to data/processed/crimes_clean.parquet") # Display confirmation message
print(f"   Rows: {len(df):,}")               # Display the final row count
print(f"   Columns: {df.shape[1]}")          # Display the final column count

✅ Cleaned dataset saved to data/processed/crimes_clean.parquet
   Rows: 3,079,424
   Columns: 24
